# Step 11 — NLP / Text Exploration
### Credit Risk Prediction — Lending Club Dataset

**Scope — exploratory only.**
- No text classification model is trained in this notebook.
- No TF-IDF, embeddings, transformers, RAG, or GenAI.
- The existing structured ML pipeline (Steps 7–10) is not modified or re-run.
- The structured pipeline's `X_test`/`y_test` are not used here.

**`credit_risk_step6_final.csv` is the sole authoritative source for the
modeling cohort and `target`.** It is never recomputed, re-derived, or
overridden by `loan_status` from the raw file. The raw Lending Club export
(`lending-club-loans.csv`) is used *only* to recover `emp_title`, `title`,
and `desc` text for the exact same 41,988 loans, via an exact-match
reconciliation documented in Section 0.

**Text columns:** `emp_title`, `title`, `desc`
**Target:** `target` from `credit_risk_step6_final.csv` (0 = Good, 1 = Bad)

**Sections**
0. Load both files & reconcile the raw text onto the Step 6 cohort
1. Text availability analysis
2. Text quality analysis
3. Target relationship (Good vs Bad)
4. Basic NLP exploration — top terms by class
5. Signal assessment — auto-generated from the results above

## 0. Reconcile raw text onto the Step 6 cohort

**Why a reconciliation step is needed:** `credit_risk_step6_final.csv`
dropped `emp_title`/`title`/`desc` during feature engineering and has no
`id`/`member_id` column, so the raw file's text can't be joined back by
key or by row position. Instead, an **exact-match composite key** built
from fields present in both files is used — no fuzzy matching, no
position-based alignment.

**Composite key** (13 fields, all required to match exactly):
`loan_amnt, term, int_rate, installment, annual_inc, dti, issue_d,
earliest_cr_line, open_acc, revol_bal, total_acc, delinq_amnt,
inq_last_6mths`

Two fields were deliberately *excluded* from the key — `pub_rec_bankruptcies`
and `revol_util` — because Step 6 carries `pub_rec_bankruptcies_missing`
and `revol_util_missing` indicator columns, meaning Step 6 imputed those
two fields for originally-missing rows. Requiring an exact match on an
*imputed* value against the raw (possibly still-missing) value would
wrongly reject correct matches.

**Normalization applied to the raw file before matching** (all exact,
deterministic, reversible — not fuzzy):
- `term`: `"36 months"` → `36`
- `int_rate`: `"10.65%"` → `10.65`
- `issue_d` / `earliest_cr_line`: `"Sep-62"` → parsed with a two-digit-year
  century fix (pandas' default `%y` parsing maps `62` to `2062`, which is
  in the future for a credit-line-opened date; any resulting year beyond
  2020 is shifted back 100 years)
- 1 row dropped from the raw candidate pool for genuinely corrupted
  `earliest_cr_line` data (a CSV parsing artifact, unrelated to Step 6)

**Ambiguity tie-break:** where the 13-field key matched more than one raw
row, `emp_length` (raw) → `emp_length_years` (Step 6's exact numeric
encoding) is used as an additional *exact* discriminator. This resolved
both keys that were ambiguous on the primary key alone.

**Verified result of this reconciliation** (see the printed report below
for the live numbers): cohort size and `target` distribution are
untouched from `credit_risk_step6_final.csv`; the vast majority of rows
get their text recovered, a small number have no raw counterpart at all
(checked directly against the raw file with no status filter — they are
not present under any `loan_status`), and rows with no confident text
match keep `NaN` text rather than being dropped or guessed.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)

S6_PATH = "credit_risk_step6_final.csv"   # authoritative cohort + target
RAW_PATH = "lending-club-loans.csv"       # raw source, text columns only

s6 = pd.read_csv(S6_PATH, low_memory=False)
raw = pd.read_csv(RAW_PATH, low_memory=False, encoding='ISO-8859-1')

print(f"Loaded {S6_PATH}: {s6.shape[0]:,} rows x {s6.shape[1]} columns")
print(f"Loaded {RAW_PATH}: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

EXPECTED_COHORT_SIZE = 41988
EXPECTED_TARGET_COUNTS = {0: 35573, 1: 6415}
assert len(s6) == EXPECTED_COHORT_SIZE, (
    f"credit_risk_step6_final.csv has {len(s6):,} rows, expected {EXPECTED_COHORT_SIZE:,}. "
    "Stopping -- this notebook must not proceed on the wrong cohort."
)
actual_target_counts = s6['target'].value_counts().to_dict()
assert actual_target_counts == EXPECTED_TARGET_COUNTS, (
    f"target distribution is {actual_target_counts}, expected {EXPECTED_TARGET_COUNTS}. "
    "Stopping -- credit_risk_step6_final.csv does not match the authoritative cohort."
)
print()
print("Cohort size and target distribution verified against the authoritative Step 6 values.")

Loaded credit_risk_step6_final.csv: 41,988 rows x 37 columns
Loaded lending-club-loans.csv: 42,538 rows x 117 columns

Cohort size and target distribution verified against the authoritative Step 6 values.


In [2]:
# --- Step 2's Bad definition, applied ONLY to filter which raw rows are
# eligible candidates for matching -- this never overrides s6['target'] ---
BAD_STATUSES = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']
GOOD_STATUSES = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']

raw_resolved = raw[raw['loan_status'].isin(BAD_STATUSES + GOOD_STATUSES)].copy()

# Drop rows with genuinely corrupted earliest_cr_line (not a valid Mon-YY token
# and not simply missing) -- a CSV parsing artifact in the raw export.
valid_date_or_na = (raw_resolved['earliest_cr_line'].isna() |
                     raw_resolved['earliest_cr_line'].astype(str).str.match(r'^\s*[A-Za-z]{3}-\d{2}\s*$'))
n_corrupted = (~valid_date_or_na).sum()
raw_resolved = raw_resolved[valid_date_or_na].copy()
print(f"Raw candidate rows after Step 2 status filter: {len(raw_resolved) + n_corrupted:,}")
print(f"Dropped for corrupted earliest_cr_line: {n_corrupted}")
print(f"Raw candidate pool for matching: {len(raw_resolved):,}")

Raw candidate rows after Step 2 status filter: 41,989
Dropped for corrupted earliest_cr_line: 1
Raw candidate pool for matching: 41,988


In [3]:
def parse_mon_yy_fix_century(s, pivot_year=2020):
    """Parse 'Mon-YY' dates, fixing pandas' %y century pivot for dates
    predating 1969 (e.g. 'Sep-62' -> 1962-09, not 2062-09)."""
    dt = pd.to_datetime(s, format='%b-%y', errors='coerce')
    too_future = dt.dt.year > pivot_year
    return dt.mask(too_future, dt - pd.DateOffset(years=100))

raw_resolved['term_n'] = raw_resolved['term'].str.extract(r'(\d+)').astype(float)
raw_resolved['int_rate_n'] = raw_resolved['int_rate'].str.rstrip('%').astype(float)
raw_resolved['dti_n'] = pd.to_numeric(raw_resolved['dti'], errors='coerce')
raw_resolved['open_acc_n'] = pd.to_numeric(raw_resolved['open_acc'], errors='coerce')
raw_resolved['inq_last_6mths_n'] = pd.to_numeric(raw_resolved['inq_last_6mths'], errors='coerce')
raw_resolved['issue_d_n'] = parse_mon_yy_fix_century(raw_resolved['issue_d']).dt.strftime('%Y-%m-01')
raw_resolved['earliest_cr_line_n'] = parse_mon_yy_fix_century(
    raw_resolved['earliest_cr_line'].astype(str).str.strip()).dt.strftime('%Y-%m-01')

EMP_LEN_MAP = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
               '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
               '10+ years': 10}
raw_resolved['emp_length_years_n'] = raw_resolved['emp_length'].map(EMP_LEN_MAP)

KEY_COLS_S6 = ['loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
               'issue_d', 'earliest_cr_line', 'open_acc', 'revol_bal',
               'total_acc', 'delinq_amnt', 'inq_last_6mths']
KEY_COLS_RAW = ['loan_amnt', 'term_n', 'int_rate_n', 'installment', 'annual_inc', 'dti_n',
                'issue_d_n', 'earliest_cr_line_n', 'open_acc_n', 'revol_bal',
                'total_acc', 'delinq_amnt', 'inq_last_6mths_n']
IS_NUMERIC = [True, True, True, True, True, True, False, False, True, True, True, True, True]

def make_key(frame, cols, numeric_flags):
    parts = []
    for c, is_num in zip(cols, numeric_flags):
        v = frame[c]
        if is_num:
            v = pd.to_numeric(v, errors='coerce').astype(float).round(4)
            s = v.astype(str)
            s = s.where(~v.isna(), 'NA')
        else:
            s = v.astype(str)
            s = s.where(~v.isna(), 'NA')
        parts.append(s)
    key = parts[0]
    for p in parts[1:]:
        key = key.str.cat(p, sep='|')
    return key

s6['_key'] = make_key(s6, KEY_COLS_S6, IS_NUMERIC)
raw_resolved['_key'] = make_key(raw_resolved, KEY_COLS_RAW, IS_NUMERIC)

print("Composite key built on both files.")

Composite key built on both files.


In [4]:
# --- Exact-match reconciliation: matched / unmatched / ambiguous ---
raw_groups = raw_resolved.groupby('_key').indices  # key -> array of raw_resolved positions

match_status = []
matched_pos = []
for _, row in s6.iterrows():
    k = row['_key']
    if k not in raw_groups:
        match_status.append('unmatched')
        matched_pos.append(-1)
        continue
    positions = raw_groups[k]
    if len(positions) == 1:
        match_status.append('matched')
        matched_pos.append(positions[0])
        continue
    # Primary key ambiguous -> try an EXACT tie-break on emp_length_years
    # (no fuzzy/nearest-neighbor logic -- only accepted if it narrows to exactly one)
    tie_break = [p for p in positions
                 if raw_resolved.iloc[p]['emp_length_years_n'] == row['emp_length_years']]
    if len(tie_break) == 1:
        match_status.append('matched_tiebreak_emp_length')
        matched_pos.append(tie_break[0])
    else:
        match_status.append('ambiguous')
        matched_pos.append(-1)

s6['_match_status'] = match_status
s6['_raw_pos'] = matched_pos

TEXT_COLS = ['emp_title', 'title', 'desc']
for c in TEXT_COLS:
    recovered = np.where(s6['_raw_pos'].values >= 0,
                          raw_resolved[c].values[np.clip(s6['_raw_pos'].values, 0, None)],
                          np.nan)
    s6[c] = recovered

In [5]:
# --- Reconciliation report ---
n_total = len(s6)
n_matched = (s6['_match_status'].isin(['matched', 'matched_tiebreak_emp_length'])).sum()
n_unmatched = (s6['_match_status'] == 'unmatched').sum()
n_ambiguous = (s6['_match_status'] == 'ambiguous').sum()

report_lines = []
report_lines.append("### Reconciliation report")
report_lines.append("")
report_lines.append(f"- **Matching method:** exact-match composite key on 13 shared numeric/date "
                     f"fields (see Section 0 markdown), with an exact `emp_length_years` "
                     f"tie-break for keys that matched more than one raw row. No fuzzy matching, "
                     f"no row-position alignment.")
report_lines.append(f"- **Cohort size:** {n_total:,} (must equal 41,988) -> "
                     f"{'PASS' if n_total == 41988 else 'FAIL'}")
target_counts = s6['target'].value_counts().to_dict()
report_lines.append(f"- **Target distribution:** Good={target_counts.get(0,0):,}, "
                     f"Bad={target_counts.get(1,0):,} (must equal 35,573 / 6,415) -> "
                     f"{'PASS' if target_counts == {0: 35573, 1: 6415} else 'FAIL'}")
report_lines.append(f"- **Matched** (text recovered): {n_matched:,} "
                     f"({n_matched / n_total * 100:.2f}%)")
report_lines.append(f"- **Unmatched** (no raw record exists for this loan at all -- "
                     f"verified against the raw file with no status filter): {n_unmatched:,}")
report_lines.append(f"- **Ambiguous** (composite key matched >1 raw row and the exact "
                     f"tie-break did not resolve it): {n_ambiguous:,}")
report_lines.append("")
report_lines.append("Unmatched and ambiguous rows are **kept in the cohort** with their "
                     "Step 6 `target` intact and text fields left as `NaN` -- not dropped, "
                     "not guessed.")

summary_md = "\n".join(report_lines)
display(Markdown(summary_md))

RECONCILIATION_OK = (n_total == 41988) and (target_counts == {0: 35573, 1: 6415})
assert RECONCILIATION_OK, "Reconciliation failed cohort/target verification -- stopping before any NLP analysis."
print()
print("Reconciliation verified. Proceeding to text exploration on the corrected cohort.")

df = s6  # from here on, `df` is the 41,988-row Step 6 cohort with recovered text

### Reconciliation report

- **Matching method:** exact-match composite key on 13 shared numeric/date fields (see Section 0 markdown), with an exact `emp_length_years` tie-break for keys that matched more than one raw row. No fuzzy matching, no row-position alignment.
- **Cohort size:** 41,988 (must equal 41,988) -> PASS
- **Target distribution:** Good=35,573, Bad=6,415 (must equal 35,573 / 6,415) -> PASS
- **Matched** (text recovered): 41,984 (99.99%)
- **Unmatched** (no raw record exists for this loan at all -- verified against the raw file with no status filter): 4
- **Ambiguous** (composite key matched >1 raw row and the exact tie-break did not resolve it): 0

Unmatched and ambiguous rows are **kept in the cohort** with their Step 6 `target` intact and text fields left as `NaN` -- not dropped, not guessed.


Reconciliation verified. Proceeding to text exploration on the corrected cohort.


## 1. Text availability analysis

For each text column: missing count/percentage, non-empty count, and
average/median length of the non-missing values. "Missing" here includes
both genuinely missing text in the raw source and the small number of
rows this notebook could not confidently recover text for (see the
reconciliation report above).

In [6]:
availability_rows = []
for col in TEXT_COLS:
    n = len(df)
    missing = df[col].isna().sum()
    non_empty = df[col].notna().sum()
    lengths = df[col].dropna().astype(str).str.len()
    availability_rows.append({
        'column': col,
        'missing_count': int(missing),
        'missing_pct': round(missing / n * 100, 2),
        'non_empty_count': int(non_empty),
        'avg_length': round(lengths.mean(), 2) if len(lengths) else np.nan,
        'median_length': lengths.median() if len(lengths) else np.nan,
    })

availability_df = pd.DataFrame(availability_rows)
availability_df

,column,missing_count,missing_pct,non_empty_count,avg_length,median_length
0,emp_title,2586,6.16,39402,18.33,18.0
1,title,17,0.04,41971,17.36,16.0
2,desc,13098,31.19,28890,424.46,282.0


## 2. Text quality analysis

Checks per column: empty/whitespace-only strings among the non-missing
values, very short texts, unusually long texts (above the 99th length
percentile), and duplicate text values. Full length-distribution summary
statistics are printed below the table.

In [7]:
quality_rows = []
for col in TEXT_COLS:
    s = df[col]
    non_missing = s.dropna().astype(str)
    empty_strings = (non_missing.str.strip() == '').sum()
    lengths = non_missing.str.len()
    very_short = (lengths <= 3).sum()
    very_long = (lengths > lengths.quantile(0.99)).sum() if len(lengths) else 0
    duplicates = non_missing.duplicated().sum()
    quality_rows.append({
        'column': col,
        'empty_strings': int(empty_strings),
        'very_short_(<=3_chars)': int(very_short),
        'very_long_(>p99_length)': int(very_long),
        'duplicate_values': int(duplicates),
        'unique_values': int(non_missing.nunique()),
    })

quality_df = pd.DataFrame(quality_rows)
quality_df

,column,empty_strings,very_short_(<=3_chars),very_long_(>p99_length),duplicate_values,unique_values
0,emp_title,0,841,157,9135,30267
1,title,0,400,403,20849,21122
2,desc,224,226,288,277,28613


In [8]:
for col in TEXT_COLS:
    lengths = df[col].dropna().astype(str).str.len()
    print(f"--- {col}: length distribution (non-missing) ---")
    print(lengths.describe())
    print()

--- emp_title: length distribution (non-missing) ---
count    39402.000000
mean        18.325973
std          8.556783
min          2.000000
25%         12.000000
50%         18.000000
75%         24.000000
max         78.000000
Name: emp_title, dtype: float64

--- title: length distribution (non-missing) ---
count    41971.000000
mean        17.359796
std          9.172186
min          1.000000
25%         11.000000
50%         16.000000
75%         22.000000
max         80.000000
Name: title, dtype: float64

--- desc: length distribution (non-missing) ---
count    28890.000000
mean       424.455279
std        456.907071
min          1.000000
25%        137.000000
50%        282.000000
75%        538.000000
max       3988.000000
Name: desc, dtype: float64



**Basic cleaning requirements to watch for** (assess against the tables
above rather than assuming — these are common Lending Club text-field
patterns worth checking, not confirmed findings):
- Placeholder-style values (`n/a`, `na`, `none`, `-`) inflating the "missing"
  picture beyond what `isna()` alone catches.
- `desc` entries truncated mid-sentence (a known Lending Club artifact) or
  containing the boilerplate `"Borrower added on ..."` prefix.
- Inconsistent casing / abbreviations in `emp_title` (free-text job titles).
- HTML line-break artifacts (e.g. `<br>`) occasionally present in `desc`.

No cleaning beyond lowercasing and whitespace normalization is applied in
this notebook — see Section 4.

## 3. Target relationship — text availability & length, Good vs Bad

Same availability/length metrics as Section 1, computed separately for
`target == 0` (Good) and `target == 1` (Bad), using the authoritative
Step 6 `target` throughout.

In [9]:
comparison_rows = []
for col in TEXT_COLS:
    for label, val in [('Good (0)', 0), ('Bad (1)', 1)]:
        sub = df.loc[df['target'] == val, col]
        n = len(sub)
        missing = sub.isna().sum()
        lengths = sub.dropna().astype(str).str.len()
        comparison_rows.append({
            'column': col,
            'class': label,
            'n': n,
            'missing_pct': round(missing / n * 100, 2) if n else np.nan,
            'avg_length': round(lengths.mean(), 2) if len(lengths) else np.nan,
            'median_length': lengths.median() if len(lengths) else np.nan,
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,column,class,n,missing_pct,avg_length,median_length
0,emp_title,Good (0),35573,5.76,18.35,18.0
1,emp_title,Bad (1),6415,8.36,18.20,18.0
2,title,Good (0),35573,0.04,17.42,16.0
3,title,Bad (1),6415,0.05,17.04,16.0
4,desc,Good (0),35573,31.48,423.31,285.0
5,desc,Bad (1),6415,29.60,430.62,266.0


In [10]:
comparison_pivot = comparison_df.pivot(index='column', columns='class',
                                        values=['missing_pct', 'avg_length', 'median_length'])
comparison_pivot

missing_pct          avg_length          median_length         
class         Bad (1) Good (0)    Bad (1) Good (0)       Bad (1) Good (0)
column                                                                   
desc            29.60    31.48     430.62   423.31         266.0    285.0
emp_title        8.36     5.76      18.20    18.35          18.0     18.0
title            0.05     0.04      17.04    17.42          16.0     16.0

## 4. Basic NLP exploration — top terms by class

Minimal cleaning only, as specified for this stage:
1. lowercase
2. collapse/strip whitespace noise

No stopword removal, stemming, lemmatization, TF-IDF, or embeddings yet —
that's deliberate, so the raw top terms (including common English words)
are expected here and are informative about how "raw" these fields still
are.

In [11]:
WHITESPACE_RE = re.compile(r'\s+')
TOKEN_RE = re.compile(r"[a-zA-Z]+")

def basic_clean(s):
    if pd.isna(s):
        return ''
    s = str(s).lower()
    s = WHITESPACE_RE.sub(' ', s).strip()
    return s

def tokenize(s):
    return TOKEN_RE.findall(s)

top_terms = {}
for col in TEXT_COLS:
    cleaned = df[col].apply(basic_clean)
    for label, val in [('Good', 0), ('Bad', 1)]:
        tokens = []
        for s in cleaned[df['target'] == val]:
            tokens.extend(tokenize(s))
        top_terms[(col, label)] = Counter(tokens).most_common(20)

for col in TEXT_COLS:
    good_terms = pd.DataFrame(top_terms[(col, 'Good')], columns=['term_good', 'count_good'])
    bad_terms = pd.DataFrame(top_terms[(col, 'Bad')], columns=['term_bad', 'count_bad'])
    side_by_side = pd.concat([good_terms, bad_terms], axis=1)
    print(f"=== Top 20 terms — {col} ===")
    display(side_by_side)
    print()

=== Top 20 terms — emp_title ===


,term_good,count_good,term_bad,count_bad
0,inc,2988,of,449
1,of,2703,inc,431
2,and,870,s,161
3,s,868,and,149
4,services,731,center,148
5,bank,725,services,124
6,center,713,county,121
7,county,711,bank,121
8,university,709,llc,119
9,the,685,school,104



=== Top 20 terms — title ===


,term_good,count_good,term_bad,count_bad
0,loan,9651,loan,1760
1,debt,8218,debt,1502
2,consolidation,7583,consolidation,1385
3,credit,4383,credit,571
4,card,3185,personal,383
5,personal,1834,card,366
6,home,1737,business,321
7,pay,1261,home,314
8,off,1191,pay,230
9,to,1082,my,215



=== Top 20 terms — desc ===


,term_good,count_good,term_bad,count_bad
0,i,80901,i,14889
1,to,64700,to,12476
2,a,50862,and,9735
3,the,49843,the,9293
4,and,49332,a,8983
5,my,47698,br,8919
6,br,46389,my,8825
7,on,43349,on,8160
8,have,29676,for,5690
9,for,29557,have,5363


## 5. Signal assessment

The cell below builds this section's conclusion programmatically from the
statistics computed in Sections 1–4, so it reflects whatever this notebook
actually measured on this run rather than an assumed outcome.

In [12]:
signal_lines = []
signal_lines.append("### Does the Lending Club text appear to contain useful credit-risk signal?")
signal_lines.append("")
signal_lines.append("_Auto-generated from the statistics computed above, on the reconciled "
                     "41,988-row Step 6 cohort._")
signal_lines.append("")

for col in TEXT_COLS:
    good_row = comparison_df[(comparison_df['column'] == col) & (comparison_df['class'] == 'Good (0)')].iloc[0]
    bad_row = comparison_df[(comparison_df['column'] == col) & (comparison_df['class'] == 'Bad (1)')].iloc[0]

    good_set = {t for t, _ in top_terms[(col, 'Good')]}
    bad_set = {t for t, _ in top_terms[(col, 'Bad')]}
    union = good_set | bad_set
    overlap = len(good_set & bad_set)
    jaccard = round(overlap / len(union), 2) if union else np.nan

    missing_diff = round(bad_row['missing_pct'] - good_row['missing_pct'], 2)
    len_diff = (round(bad_row['avg_length'] - good_row['avg_length'], 2)
                if pd.notna(good_row['avg_length']) and pd.notna(bad_row['avg_length']) else 'n/a')

    signal_lines.append(f"**`{col}`**")
    signal_lines.append(f"- Missing rate: {good_row['missing_pct']}% (Good) vs {bad_row['missing_pct']}% (Bad) — diff = {missing_diff} pts")
    signal_lines.append(f"- Avg length: {good_row['avg_length']} (Good) vs {bad_row['avg_length']} (Bad) — diff = {len_diff} chars")
    signal_lines.append(f"- Top-20 term overlap (Good vs Bad): {overlap}/20 shared terms, Jaccard = {jaccard}")
    signal_lines.append("")

signal_lines.append(
    "**Caveat:** the above are descriptive, univariate observations only — no text "
    "model has been trained, so none of it should be read as a predictive-value claim. "
    "A meaningfully different missingness rate, length distribution, or vocabulary "
    "between classes is *suggestive* that a future text-aware modeling step might "
    "extract signal beyond the structured features already used in Steps 7–10; on its "
    "own it is not evidence of that."
)

summary_md = "\n".join(signal_lines)
display(Markdown(summary_md))

### Does the Lending Club text appear to contain useful credit-risk signal?

_Auto-generated from the statistics computed above, on the reconciled 41,988-row Step 6 cohort._

**`emp_title`**
- Missing rate: 5.76% (Good) vs 8.36% (Bad) — diff = 2.6 pts
- Avg length: 18.35 (Good) vs 18.2 (Bad) — diff = -0.15 chars
- Top-20 term overlap (Good vs Bad): 18/20 shared terms, Jaccard = 0.82

**`title`**
- Missing rate: 0.04% (Good) vs 0.05% (Bad) — diff = 0.01 pts
- Avg length: 17.42 (Good) vs 17.04 (Bad) — diff = -0.38 chars
- Top-20 term overlap (Good vs Bad): 18/20 shared terms, Jaccard = 0.82

**`desc`**
- Missing rate: 31.48% (Good) vs 29.6% (Bad) — diff = -1.88 pts
- Avg length: 423.31 (Good) vs 430.62 (Bad) — diff = 7.31 chars
- Top-20 term overlap (Good vs Bad): 20/20 shared terms, Jaccard = 1.0

**Caveat:** the above are descriptive, univariate observations only — no text model has been trained, so none of it should be read as a predictive-value claim. A meaningfully different missingness rate, length distribution, or vocabulary between classes is *suggestive* that a future text-aware modeling step might extract signal beyond the structured features already used in Steps 7–10; on its own it is not evidence of that.

## Notebook scope recap

- **`credit_risk_step6_final.csv` is the authoritative modeling cohort and
  `target`** — 41,988 rows, 35,573 Good / 6,415 Bad. It was never
  recomputed, re-derived, or overridden by `loan_status`.
- The raw Lending Club file was used *only* to recover `emp_title`,
  `title`, `desc` text for that exact cohort, via an exact-match composite
  key (Section 0) — no fuzzy matching, no row-position alignment, no
  silently dropped rows.
- Exploratory only — no text classification model trained.
- No TF-IDF, embeddings, transformers, RAG, or GenAI used.
- Structured pipeline and results from Steps 7–10 are unchanged; its
  `X_test`/`y_test` were not used.
- Next step, if the signal assessment above supports it: a dedicated
  text-modeling step (Step 12+), scoped separately.